# InternScenes scene_id -> USD

Converts a scene into an Isaac Sim-compatible USD file given a `scene_id`, fetching only that scene's data.

`data/` (symlinked to `external-lib/InternScenes/data`) holds a curated, per-scene subset of the dataset: `Layout_info/` (scene metadata) and `asset_library/` (GLB assets). The full `InternRobotics/InternScenes` HF repo is ~2.46 TB and must never be mirrored in full -- see the download section below.

Run the cells in order; change `SCENE_ID` wherever it appears (e.g. `scene0000_01`, `scene0001_00`).

In [8]:
from pathlib import Path
import sys

PROJECT_ROOT = Path('/home/snt/projects/AgenticMemoryNav')
DATA_ROOT = PROJECT_ROOT / 'data'
# Curated per-scene subset (Layout_info/ + asset_library/) lives directly under DATA_ROOT.
# NOTE: DATA_ROOT / 'InternScenes' is a *separate*, full raw HF clone (100+ GB) -- never use that path here.
DATASET_ROOT = DATA_ROOT
INTERNSCENES_SRC = PROJECT_ROOT / 'external-lib' / 'InternScenes'
ISAAC_PY = Path('/home/snt/isaacsim/python.sh')
INTERNSCENES_VENV_PY = PROJECT_ROOT / '.internScenes-venv' / 'bin' / 'python3'

for p in [str(INTERNSCENES_SRC), str(INTERNSCENES_SRC / 'InternScenes')]:
    if p not in sys.path:
        sys.path.insert(0, p)

print('PROJECT_ROOT =', PROJECT_ROOT)
print('DATASET_ROOT =', DATASET_ROOT)
print('INTERNSCENES_SRC =', INTERNSCENES_SRC)
print('ISAAC_PY exists =', ISAAC_PY.exists())
print('INTERNSCENES_VENV_PY exists =', INTERNSCENES_VENV_PY.exists())

print('layout folder exists =', (DATASET_ROOT / 'Layout_info').exists())
print('asset library exists =', (DATASET_ROOT / 'asset_library').exists())

PROJECT_ROOT = /home/snt/projects/AgenticMemoryNav
DATASET_ROOT = /home/snt/projects/AgenticMemoryNav/data
INTERNSCENES_SRC = /home/snt/projects/AgenticMemoryNav/external-lib/InternScenes
ISAAC_PY exists = True
INTERNSCENES_VENV_PY exists = True
layout folder exists = True
asset library exists = True


## Download only what a scene needs (recommended)

The full `InternRobotics/InternScenes` dataset is ~2.46 TB, so never `git clone` or `snapshot_download` the whole repo (the raw clone under `data/InternScenes/` alone is already 150+ GB on this machine and disk is nearly full). Instead, fetch data **per scene**:

1. **Layout metadata** -- `Layout_info.tar.gz` (~2.8 GB, covers *all* datasets/scenes) is downloaded once and cached under `data/.hf_layout/`. A single scene's `layout.json` + `StructureMesh/*.glb` are extracted from that local cache on demand, with **no extra network call**.
2. **3D asset GLBs** -- `download_scene_assets.py` reads a scene's `layout.json`, resolves only the unique `model_uid`s it references, and downloads just those files from Hugging Face (a typical scene needs ~10-100 small GLBs, tens to a few hundred MB, instead of the full multi-terabyte `asset_library`).

Run the next two cells for any `scene_id` before composing/converting it.

In [9]:
import subprocess
import tarfile

LAYOUT_TAR = DATA_ROOT / '.hf_layout' / 'Layout_info.tar.gz'
DOWNLOAD_ASSETS_SCRIPT = INTERNSCENES_SRC / 'download_scene_assets.py'


def ensure_layout_for_scene(scene_id: str, dataset: str = 'scannet') -> Path:
    """Extract one scene's layout.json/StructureMesh from the cached Layout_info tar (no network)."""
    target_dir = DATA_ROOT / 'Layout_info' / dataset / scene_id
    layout_path = target_dir / 'layout.json'
    if layout_path.exists():
        return layout_path

    if not LAYOUT_TAR.exists():
        raise FileNotFoundError(
            f'{LAYOUT_TAR} not found. Fetch it once (whole-file, ~2.8 GB, covers every scene) with:\n'
            f'  huggingface-cli download InternRobotics/InternScenes Layout_info.tar.gz '
            f'--repo-type dataset --local-dir {DATA_ROOT / ".hf_layout"}'
        )

    prefix = f'Layout_info/{dataset}/{scene_id}/'
    with tarfile.open(LAYOUT_TAR, 'r:gz') as tf:
        members = [m for m in tf.getmembers() if m.name.startswith(prefix)]
        if not members:
            raise FileNotFoundError(f'No entries for {prefix} inside {LAYOUT_TAR}. Check the scene id/dataset.')
        tf.extractall(path=DATA_ROOT, members=members)

    if not layout_path.exists():
        raise FileNotFoundError(f'Extraction finished but {layout_path} is still missing')
    return layout_path


def download_scene_assets_for_id(
    scene_id: str,
    dataset: str = 'scannet',
    no_archives: bool = False,
    with_objaverse: bool = False,
    dry_run: bool = False,
) -> subprocess.CompletedProcess:
    """Fetch only the GLB assets that one scene's layout.json references (per-scene, not the full library)."""
    layout_path = ensure_layout_for_scene(scene_id, dataset)
    cmd = [str(INTERNSCENES_VENV_PY), str(DOWNLOAD_ASSETS_SCRIPT), str(layout_path), '--dest', str(DATA_ROOT)]
    if no_archives:
        cmd.append('--no-archives')
    if with_objaverse:
        cmd.append('--with-objaverse')
    if dry_run:
        cmd.append('--dry-run')
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print('STDERR:\n', result.stderr[-2000:])
    return result


## Convert scene id(s) to USD

`convert_scene_id_to_usd` ensures the layout + assets are present (per-scene, via the helpers above), then runs the headless Isaac Sim converter on the composed GLB.

In [10]:
import subprocess
from pathlib import Path
from typing import Optional, Tuple

def compose_scene_id(scene_id: str, dataset: str = 'scannet') -> Path:
    """Compose one scene's layout.json + assets into a single GLB (via compose_one.py)."""
    intern_src = PROJECT_ROOT / 'external-lib' / 'InternScenes'
    glb_path = PROJECT_ROOT / 'tutorial' / 'examples' / 'composed_scenes' / dataset / scene_id / 'glb_scene.glb'
    if glb_path.exists():
        return glb_path

    cmd = [str(INTERNSCENES_VENV_PY), str(intern_src / 'compose_one.py'), f'{dataset}/{scene_id}']
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=str(intern_src))
    print(result.stdout[-1500:])
    if result.returncode != 0:
        print('STDERR:')
        print(result.stderr[-1500:])
    if not glb_path.exists():
        raise FileNotFoundError(f'Composition did not produce {glb_path}')
    return glb_path


def convert_scene_id_to_usd(scene_id: str, dataset: str = 'scannet', output_root: Optional[Path] = None) -> Tuple[Path, bool, str]:
    """Convert one composed InternScenes GLB into an Isaac Sim-compatible USD file."""
    project_root = PROJECT_ROOT
    intern_src = project_root / 'external-lib' / 'InternScenes'
    isaac_py = Path('/home/snt/isaacsim/python.sh')

    if output_root is None:
        output_root = project_root / 'issacsim-assets'

    ensure_layout_for_scene(scene_id, dataset)
    download_scene_assets_for_id(scene_id, dataset)
    glb_path = compose_scene_id(scene_id, dataset)

    usd_path = output_root / scene_id / 'usd' / 'scene.usd'
    usd_path.parent.mkdir(parents=True, exist_ok=True)

    cmd = [
        str(isaac_py),
        str(intern_src / 'glb2usd_headless.py'),
        '--file', str(glb_path),
        '--out', str(usd_path),
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    ok = result.returncode == 0 and usd_path.exists()
    print(f'[{scene_id}] rc={result.returncode}, usd_exists={usd_path.exists()}')
    if not ok:
        print('STDOUT:')
        print(result.stdout[-1500:])
        print('STDERR:')
        print(result.stderr[-1500:])
    return usd_path, ok, (result.stderr.strip() if result.stderr else 'ok')

# Example usage:
usd_path, ok, msg = convert_scene_id_to_usd('scene0003_00')
print(usd_path, ok, msg)

scene: /home/snt/projects/AgenticMemoryNav/data/Layout_info/scannet/scene0003_00/layout.json
objects: 20  unique assets: 20
per-object GLBs: 9  archives: 1
downloaded into /home/snt/projects/AgenticMemoryNav/data

process_instance 5 done
process_instance 7 done
process_instance 0 done
process_instance 9 done
process_instance 10 done
process_instance 6 done
process_instance 11 done
process_instance 3 done
process_instance 12 done
process_instance 4 done
process_instance 13 done
process_instance 17 doneprocess_instance 8 done

process_instance 19 done
process_instance 16 done
process_instance 15 done
process_instance 18 done
process_instance 2 done
process_instance 1 done
process_instance 14 done
Composed glb scene has been saved to /home/snt/projects/AgenticMemoryNav/tutorial/examples/composed_scenes/scannet/scene0003_00/glb_scene.glb
GLB_PATH=/home/snt/projects/AgenticMemoryNav/tutorial/examples/composed_scenes/scannet/scene0003_00/glb_scene.glb
GLB_EXISTS=True

[scene0003_00] rc=0, us

In [13]:
import pandas as pd

scene_ids = ['scene0000_01']  # add more scene ids here
summary = []

for sid in scene_ids:
    try:
        usd_path, ok, msg = convert_scene_id_to_usd(sid, dataset='scannet')
        glb_path = next((p for p in [
            PROJECT_ROOT / 'tutorial' / 'examples' / 'composed_scenes' / 'scannet' / sid / 'glb_scene.glb',
            INTERNSCENES_SRC / 'InternScenes' / 'tutorial' / 'examples' / 'composed_scenes' / 'scannet' / sid / 'glb_scene.glb',
            INTERNSCENES_SRC / 'tutorial' / 'examples' / 'composed_scenes' / 'scannet' / sid / 'glb_scene.glb',
        ] if p.exists()), None)
        summary.append({
            'scene_id': sid,
            'glb_path': str(glb_path) if glb_path else None,
            'usd_path': str(usd_path),
            'success': ok,
            'message': msg,
        })
    except Exception as e:
        summary.append({
            'scene_id': sid,
            'glb_path': None,
            'usd_path': None,
            'success': False,
            'message': str(e),
        })

summary_df = pd.DataFrame(summary)
print(summary_df)

summary_path = PROJECT_ROOT / 'issacsim-assets' / 'scene_conversion_summary.csv'
summary_path.parent.mkdir(parents=True, exist_ok=True)
summary_df.to_csv(summary_path, index=False)
print('saved conversion summary ->', summary_path)



STDERR:
 /home/snt/projects/AgenticMemoryNav/.internScenes-venv/bin/python3: can't open file '/home/snt/projects/AgenticMemoryNav/external-lib/InternScenes/download_scene_assets.py': [Errno 2] No such file or directory

       scene_id glb_path usd_path  success  \
0  scene0000_01     None     None    False   

                                             message  
0  No composed GLB found for scene0000_01. Compos...  
saved conversion summary -> /home/snt/projects/AgenticMemoryNav/issacsim-assets/scene_conversion_summary.csv


## Render a top-down preview

Quick visual sanity check of the converted USD.

In [ ]:
from pathlib import Path
import numpy as np

SCENE_ID = scene_ids[0]  # pick the scene converted above to preview
USD_OUT_PATH = PROJECT_ROOT / 'issacsim-assets' / SCENE_ID / 'usd' / 'scene.usd'
RENDER_OUT = PROJECT_ROOT / 'issacsim-assets' / SCENE_ID / 'render' / 'topdown.png'
RENDER_OUT.parent.mkdir(parents=True, exist_ok=True)

if not USD_OUT_PATH.exists():
    raise FileNotFoundError(f'USD file not found; convert {SCENE_ID} first: {USD_OUT_PATH}')

print('Rendering top-down preview for', SCENE_ID)
print('USD input =', USD_OUT_PATH)
print('output PNG =', RENDER_OUT)

try:
    from isaacsim import SimulationApp
    from isaacsim.core.api import World
    from isaacsim.sensors.camera import Camera
    from omni.usd import get_context
    from pxr import Gf
    from PIL import Image

    app = SimulationApp({"headless": True})
    try:
        world = World(stage_units_in_meters=1.0)
        ctx = get_context()
        ctx.open_stage(str(USD_OUT_PATH))

        camera = Camera(prim_path='/World/topdown_camera', resolution=(768, 768), frequency=10)
        camera.initialize()
        camera.set_focal_length(24.0)
        camera.add_distance_to_image_plane_to_frame()

        camera.set_local_pose(
            translation=np.array([0.0, 0.0, 18.0], dtype=np.float32),
            orientation=Gf.Quatd(0.70710678, 0.70710678, 0.0, 0.0),
        )
        world.reset()
        world.step(render=True)

        rgba = camera.get_rgba()
        if rgba is None or rgba.size == 0:
            raise RuntimeError('Camera render returned no pixels.')

        image = rgba[:, :, :3].astype(np.uint8)
        Image.fromarray(image).save(RENDER_OUT)
        print('Saved top-down render:', RENDER_OUT)
        print('PNG exists =', RENDER_OUT.exists(), 'size_bytes =', RENDER_OUT.stat().st_size)
    finally:
        app.close()
except Exception as e:
    print('Top-down render failed:', type(e).__name__, e)
    raise